### Load the AI Core service key

In [ ]:
import dotenv from "dotenv";
dotenv.config();

const serviceKey = JSON.parse(process.env.AICORE_SERVICE_KEY);

const AI_API_URL = serviceKey.serviceurls.AI_API_URL;
const clientid = serviceKey.clientid;
const clientsecret = serviceKey.clientsecret;
const authUrl = serviceKey.url;

### Resource Group

In [ ]:
import { ResourceGroupApi } from '@sap-ai-sdk/ai-api';

const RESOURCE_GROUP  = 'rg-test2' // Please change to your desired ID

// Create resource group using ResourceGroupApi
async function createResourceGroup() {
    try {
      const response = await ResourceGroupApi.
        kubesubmitV4ResourcegroupsCreate({
            resourceGroupId: RESOURCE_GROUP,
            labels: [
                {
                key: 'ext.ai.sap.com/document-grounding',
                value: 'true',
                }
            ]
        }).execute();
        return response.resourceGroupId;
    } catch (error: any) {
      console.error('Error while creating Resource Group:', error.stack);
    }
}

const resourceGroupId = await createResourceGroup();
console.log("Created Resource Group with ID: ", resourceGroupId)


Created Resource Group with ID:  rg-test2


### Create a New Orchestration Configuration
In this step, we define a function to create an orchestration configuration using the ConfigurationApi from the SAP AI SDK. This configuration integrates various parameters needed for orchestration, such as the executable ID and scenario ID.

Key Points:

ConfigurationApi: Provides methods for interacting with the SAP AI SDK's configuration services.

parameterBindings: Specifies the parameters used for orchestration.

In [11]:
import { ConfigurationApi } from '@sap-ai-sdk/ai-api';

async function createOrchestrationConfiguration(resourceGroupId: string) {
  const requestBody = {
    name: 'orchestration-config',
    executableId: 'orchestration',
    scenarioId: 'orchestration',
    parameterBindings: [
      { key: 'modelFilterList', value: 'null' },
      { key: 'modelFilterListType', value: 'allow' }
    ],
    inputArtifactBindings: []
  };

  try {
    const responseData = await ConfigurationApi
      .configurationCreate(requestBody, {
        'AI-Resource-Group': resourceGroupId
      })
      .execute();

    return responseData;
  } catch (error) {
    if (error.response) {
      console.error(error.response.status);
      console.error(error.response.data);
    } else {
      console.error(error.message);
    }
  }
}

// usage
const orchestrationConfig = await createOrchestrationConfiguration(resourceGroupId);
console.log(orchestrationConfig);


{
  id: "a4bfd193-6745-44ee-aa56-f5643830e6fc",
  message: "Configuration created"
}


### Deployment of orchestration
This step involves creating a deployment using the specified configuration and resource group. The deployment is handled via the DeploymentApi, which streamlines the process of activating the orchestration setup.

Key Points:

DeploymentApi: Used for initiating the deployment based on the given configuration.

createDeployment Function: This function handles the API call to create the deployment.

In [12]:
import { DeploymentApi } from '@sap-ai-sdk/ai-api';
import type { AiDeploymentCreationResponse } from '@sap-ai-sdk/ai-api';

/**
 * Create a deployment using the configuration specified by configurationId.
 * @param configurationId - ID of the configuration to be used.
 * @param resourceGroup - AI-Resource-Group where the resources are available.
 * @returns Deployment creation response with 'targetStatus': 'RUNNING'.
 */
export async function createDeployment(
  configurationId: string,
  resourceGroup: string
): Promise<AiDeploymentCreationResponse> {
  return DeploymentApi.deploymentCreate(
    { configurationId },
    { 'AI-Resource-Group': resourceGroup }
  ).execute();
}
/**
 * Deploy the orchestration using the given configuration ID.
 * @param resourceGroup - AI-Resource-Group where the resources are available.
 * @returns A message indicating the result of the deployment operation.
 */
export async function deployOrchestration(
  resourceGroup: string
): Promise<string> {
  // Fetch the configuration ID (can be retrieved or passed dynamically)
   const configurationId = orchestrationConfig.id;

  try {
    // Step: Create deployment using the configuration ID
    const response = await createDeployment(configurationId, resourceGroup);
    // console.log(`Orchestration deployment created with ID: ${response.id}`);
    return `Orchestration deployment created with ID: ${response.id}`;
  } catch (error) {
    console.error('Error creating orchestration deployment:', error);
    return 'Failed to create orchestration deployment.';
  }
}

In [13]:
// usage to deploy orchestration
(async () => { 
    try {
      const result = await deployOrchestration(resourceGroupId);
      console.log(result); // Outputs deployment creation response
    } catch (error) {
      console.error('Error executing orchestration deployment:', error);
    }
  })();

Promise { <pending> }

Orchestration deployment created with ID: d55308d355e4a756


Here, you are explicitly defining the orchestration service deployment URL (orchestration_service_url) which points to your deployed LLM configuration. This URL is used to send inference requests (like prompt executions) to the SAP AI Core Orchestration.

### Generic Secret

#### In this tutorial, we're demonstrating how to create a vector knowledge base by connecting either SharePoint or AWS S3 as the document source—multiple options are supported and optional based on your setup.

#### creating knowledge base using Sharepoint - option 1

This step specifically creates a secret in SAP AI Core that stores Base64-encoded credentials for SharePoint access, securely enabling document grounding workflows via Microsoft Graph.

In [ ]:
import { SecretApi } from '@sap-ai-sdk/ai-api';

// Create Secret using SecretApi
async function createGenericSecret() {
    try {
        const response = await SecretApi.kubesubmitV4GenericSecretsCreate({
          name: 'canary-rg1-secret',
          data: {
            type: 'SFRUUA==',
            description: '<DESCRIPTION_OF_GENERIC_SECRET>',
            clientId: '<CLIENT_ID>',
            authentication: '<AUTHENTICATION>',
            tokenServiceURL: '<TOKEN_SERVICE_URL>',
            password: '<PASSWORD>',
            proxyType: '<PROXY>',
            url: '<URL>',
            tokenServiceURLType: '<TOKEN_SERVICE_URL_TYPE>',
            user: '<USER>',
            clientSecret: '<CLIENT_SECRET>',
            scope: '<SCOPE>'
          },
          labels: [
            {
              key: 'ext.ai.sap.com/document-grounding',
              value: 'true',
            },
          ],
        }).execute();
        return response;
    } catch (error: any) {
        console.error('Error while creating Resource Group:', error.stack);
    }
}

const secret = await createGenericSecret();
console.log(secret?.message)

#### creating knowledge base using AWS S3 - Option 2

Alternatively, instead of SharePoint, we can use AWS S3 as a document repository for grounding. In the example below, we securely store credentials as a secret named aws-s3-secret that will later be referenced in the pipeline creation.

This makes it clear that both SharePoint and AWS S3 are optional approaches and interchangeable based on the user’s infrastructure.

In [ ]:
import { SecretApi } from '@sap-ai-sdk/ai-api';

async function createS3GenericSecret() {
  try {
    const response = await SecretApi.kubesubmitV4GenericSecretsCreate(
      {
        name: 's3-grounding-secret',
        data:  {
              description: "<description of generic secret>",
              url: "<url>",
              authentication: "Tm9BdXRoZW50aWNhdGlvbg==",
              access_key_id: "<access key id>",
              secret_access_key: "<secret access key>",
              bucket: "<bucket>",
              region: "<region>",
              host: "<host>",
              username: "<username>" ,
              type: "SFRUUA==",
              proxyType: "<PROXY>" 
            },
        labels: [
          {
            key: 'ext.ai.sap.com/document-grounding',
            value: 'true'
          },
          {
            key: 'ext.ai.sap.com/documentRepositoryType',
            value: 'S3'
          }
        ]
      },
      {
        'AI-Resource-Group': '<resource_group>'
      }
    ).execute();

    console.log('✅ S3 Generic Secret created:', response.name);
    return response;
  } catch (error: any) {
    console.error(
      '❌ Error while creating S3 Generic Secret:',
      error.cause?.response?.data || error.message
    );
  }
}

await createS3GenericSecret();


✅ S3 Generic Secret created: s3-grounding-secret


{ message: "secret has been created", name: "s3-grounding-secret" }

### Pipeline Creation

#### Pipeline creation using sharepoint - option 1
In this step, we are creating a document grounding pipeline using SharePoint as the knowledge source. The pipeline connects to the document repository defined in the SharePoint site using the previously created secret 

In [ ]:
// Request body for pipeline creation request
const pipelineRequest: PipelinePostRequst = {
  type: 'MSSharePoint',
  configuration: {
    destination: 'canary-rg1-secret',
    sharePoint: {
      site: {
        name: '<sharepoint site name>',
        includePaths: ['/<folder name>']
      }
    }
  }
};

// Create the pipeline
const pipeline = await PipelinesApi.createPipeline(pipelineRequest, {
  'AI-Resource-Group': RESOURCE_GROUP
}).execute();

console.log('Created Pipeline with ID: ', pipeline.pipelineId);

#### Pipeline creation using AWS S3 - option 2
Once the secret (aws-s3-secret) is created, we can now configure the document grounding pipeline using AWS S3 as the data source. This example shows how to set up a pipeline by referencing the created secret. The pipeline will extract and prepare documents from the specified S3 bucket for grounding.

🔄 You can follow a similar flow for SharePoint or other supported sources — choosing between SharePoint and S3 is flexible based on your document storage setup.

In [ ]:

import { PipelinesApi } from '@sap-ai-sdk/document-grounding';

const RESOURCE_GROUP = resourceGroupId;

const s3PipelineRequest: CreatePipeline = {
  type: 'S3',
  configuration: {
    destination: 's3-grounding-secret'
  }
};

try {
  const pipeline = await PipelinesApi.createPipeline(
    s3PipelineRequest,
    { 'AI-Resource-Group': RESOURCE_GROUP }
  ).execute();

  console.log('✅ S3 Pipeline created successfully');
  console.log('Pipeline ID:', pipeline.pipelineId);
} catch (error: any) {
  console.error('❌ Pipeline creation failed');
  console.error(error.cause?.response?.data || error.message);
}


✅ S3 Pipeline created successfully
Pipeline ID: 914b1a69-0413-4dfc-8582-5f9adf3a1fa5


#### Set Up the Orchestration Service

Now that we have our document grounding pipeline ready, we can configure the LLM Orchestration Service to process incoming user queries in context.

We define a system message to describe the business scenario for the LLM — in this case, a Facility Solutions Company offering property maintenance and support services. The prompt template includes placeholders for the user’s query and the grounded document context (retrieved from S3 or SharePoint), making the responses personalized and context-aware.

💡 This setup ensures that the LLM generates accurate, domain-specific, and grounded responses using the extracted content from your enterprise documents.

In [ ]:
import {
  OrchestrationClient,  buildDocumentGroundingConfig,  buildAzureContentSafetyFilter} 
  from '@sap-ai-sdk/orchestration';

// ---------------------------
// Initialize Orchestration Client
// ---------------------------
const orchestrationClient = new OrchestrationClient({
  promptTemplating: {
    model: {
      name: 'gpt-4o',
      params: {
        max_completion_tokens: 200,
        temperature: 0
      }
    }
  },
  grounding: buildDocumentGroundingConfig({
    placeholders: {
      input: ['groundingRequest'],
      output: 'groundingOutput' 
    },
    filters: [
      {
        id: 'filter1',
        data_repositories: ['a0165**************55f'],
        data_repository_type: 'vector',
        search_config: { max_chunk_count: 20 }
      }
    ]
  }),
  filtering: {
    input: {
      filters: [
        buildAzureContentSafetyFilter({
          Hate: 'ALLOW_SAFE_LOW',
          Violence: 'ALLOW_SAFE_LOW'
        })
      ]
    },
    output: {
      filters: [
        buildAzureContentSafetyFilter({
          Hate: 'ALLOW_SAFE_LOW',
          Violence: 'ALLOW_SAFE_LOW'
        })
      ]
    }
  }
},
{resourceGroup:resourceGroupId}

);

// ---------------------------
// Send Chat Completion Request
// ---------------------------
const response = await orchestrationClient.chatCompletion({
  messages: [
    {
            role: 'system',
            content: `Facility Solutions Company provides services to luxury residential complexes, apartments,
individual homes, and commercial properties such as office buildings, retail spaces, industrial facilities, and educational institutions.
Customers are encouraged to reach out with maintenance requests, service deficiencies, follow-ups, or any issues they need by email.`
          },
          {
            role: 'user',
            content: `You are a helpful assistant for any queries.
Answer the request by providing relevant answers that fit the request.
Request: {{ ?groundingRequest }}
Context: {{ ?groundingOutput }}`
          }
  ],
  placeholderValues: {
    groundingRequest: 'Is there any complaint from customers?'
  }
},
);
console.log(response.getContent());

Based on the provided context, there are several complaints from customers regarding various services. Here are the specific complaints:

1. **Window Cleaning Service Oversight**: Mark Phillips reported that the windows in the east wing of the Riverfront Business Complex were missed during the last cleaning service, and this has been a recurring issue.

2. **Landscaping Service Issues**: James Anderson mentioned problems with the recent landscaping service at Crestview Gardens Apartments, where the shrubs were not trimmed properly, and debris was left behind.

3. **Missed Cleaning in Conference Room**: Michael Nguyen noted that the conference room was missed during the cleaning service at their main office on Elm Street.

4. **Malfunctioning Elevator**: Raj Patel is following up on a maintenance request regarding a malfunctioning elevator in Building B at Oakwood Corporate Center.

5. **Heating System Malfunction**: Emily Carter reported an urgent issue with the heating system in her a

In [15]:
response 

OrchestrationResponse {
  rawResponse: {
    status: 200,
    statusText: "OK",
    headers: Object [AxiosHeaders] {
      date: "Thu, 05 Feb 2026 10:07:32 GMT",
      "content-type": "application/json",
      "content-length": "16942",
      "x-upstream-service-time": "2525"
    },
    config: {
      transitional: {
        silentJSONParsing: true,
        forcedJSONParsing: true,
        clarifyTimeoutError: false
      },
      adapter: [ "xhr", "http", "fetch" ],
      transformRequest: [ [Function: transformRequest] ],
      transformResponse: [ [Function: transformResponse] ],
      timeout: 0,
      xsrfCookieName: "XSRF-TOKEN",
      xsrfHeaderName: "X-XSRF-TOKEN",
      maxContentLength: -1,
      maxBodyLength: -1,
      env: {
        FormData: [Function: FormData] {
          LINE_BREAK: "\r\n",
          DEFAULT_CONTENT_TYPE: "application/octet-stream"
        },
        Blob: [class Blob]
      },
      validateStatus: [Function: validateStatus],
      headers: Object [A